In [0]:
spark.sql("DROP TABLE IF EXISTS subscription_catalog.gold_schema.fct_opportunities")

In [0]:
%sql
CREATE OR REPLACE TABLE `subscription_catalog`.gold_schema.fact_opportunities AS
SELECT 
  opportunity_id,
  customer_id,
  product_id,
  employee_id,
  customer_country,
  start_date,
  end_date,
  created_timestamp,
  contract_term,
  close_status,
  transaction_currency,
  revenue_amount,
  list_price,
  revenue_in_gbp,
  DATEDIFF(end_date, start_date) as contract_duration_days
FROM `subscription_catalog`.silver_schema.sl_opportunity_main

In [0]:
%sql
CREATE OR REPLACE TABLE `subscription_catalog`.gold_schema.dim_geography AS
SELECT DISTINCT
  customer_country,
  transaction_currency
FROM `subscription_catalog`.silver_schema.sl_opportunity_main
ORDER BY customer_country

In [0]:
%sql
CREATE OR REPLACE TABLE `subscription_catalog`.gold_schema.dim_contract_type AS
SELECT DISTINCT
  contract_term,
  CASE 
    WHEN contract_term = 'Monthly' THEN 1
    WHEN contract_term = 'Yearly' THEN 12
    ELSE 0
  END as months_in_term
FROM `subscription_catalog`.silver_schema.sl_opportunity_main
ORDER BY contract_term

In [0]:
%sql
CREATE OR REPLACE TABLE `subscription_catalog`.gold_schema.dim_close_status AS
SELECT DISTINCT
  close_status,
  CASE 
    WHEN close_status = 'WON' THEN 1
    WHEN close_status = 'ACTIVE' THEN 1
    WHEN close_status = 'LOST' THEN 0
    ELSE NULL
  END as is_successful
FROM `subscription_catalog`.silver_schema.sl_opportunity_main
ORDER BY close_status

In [0]:
%sql
CREATE OR REPLACE TABLE `subscription_catalog`.gold_schema.dim_date AS
SELECT DISTINCT
  date_value,
  YEAR(date_value) as year,
  QUARTER(date_value) as quarter,
  MONTH(date_value) as month,
  DAYOFMONTH(date_value) as day,
  DAYOFWEEK(date_value) as day_of_week,
  WEEKOFYEAR(date_value) as week_of_year,
  DATE_FORMAT(date_value, 'MMMM') as month_name,
  DATE_FORMAT(date_value, 'EEEE') as day_name
FROM (
  SELECT start_date as date_value FROM `subscription_catalog`.silver_schema.sl_opportunity_main
  UNION
  SELECT end_date as date_value FROM `subscription_catalog`.silver_schema.sl_opportunity_main
  UNION
  SELECT DATE(created_timestamp) as date_value FROM `subscription_catalog`.silver_schema.sl_opportunity_main
)
WHERE date_value IS NOT NULL
ORDER BY date_value

In [0]:
%sql
CREATE OR REPLACE TABLE `subscription_catalog`.gold_schema.dim_customer AS
SELECT DISTINCT
  customer_id,
  COUNT(*) OVER (PARTITION BY customer_id) as total_opportunities
FROM `subscription_catalog`.silver_schema.sl_opportunity_main
ORDER BY customer_id

In [0]:
%sql
CREATE OR REPLACE TABLE `subscription_catalog`.gold_schema.dim_product AS
SELECT DISTINCT
  product_id,
  AVG(list_price) as avg_list_price,
  COUNT(*) as times_sold
FROM `subscription_catalog`.silver_schema.sl_opportunity_main
GROUP BY product_id
ORDER BY product_id

In [0]:
%sql
CREATE OR REPLACE TABLE `subscription_catalog`.gold_schema.dim_employee AS
SELECT DISTINCT
  employee_id,
  COUNT(*) as total_opportunities_handled,
  SUM(CASE WHEN close_status = 'WON' THEN 1 ELSE 0 END) as won_opportunities,
  ROUND(SUM(CASE WHEN close_status = 'WON' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as win_rate_pct
FROM `subscription_catalog`.silver_schema.sl_opportunity_main
GROUP BY employee_id
ORDER BY employee_id